# 02 — Customer, delivery and satisfaction analysis

Builds on [`01_data_quality_eda.ipynb`](01_data_quality_eda.ipynb), which
established that the data supports these measures.

Covers README questions **Q4–Q7**: how new and repeat customers differ, how
often deliveries miss their promised date, whether delivery performance tracks
review scores, and which segments are worth retaining.

Every figure referenced here is regenerated by `python scripts/run_analysis.py`.

In [1]:
import sqlite3
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

DB = ROOT / "data" / "olist.db"
assert DB.exists(), "run `python scripts/build_db.py` first, then `python scripts/run_analysis.py`"
conn = sqlite3.connect(DB)
q = lambda sql: pd.read_sql_query(sql, conn)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("connected to", DB.relative_to(ROOT))

connected to data/olist.db


## 1. The repeat-purchase problem

Start with the measure that decides the entire customer strategy.

In [2]:
pd.read_csv(ROOT / "outputs" / "tables" / "customer_type_summary.csv")

,customer_type,customers,pct_of_customers,orders,gmv,pct_of_gmv,gmv_per_customer,orders_per_customer,aov
0,one-time,90557,97.0,90557,12493089.36,94.49,137.96,1.00,137.96
1,repeat,2801,3.0,5921,728408.75,5.51,260.05,2.11,123.02


**3.0% of customers ever place a second delivered order.** For an
e-commerce marketplace this is very low — healthy marketplaces typically run
20–40%.

Note what the repeat customers who *do* exist look like: 2.11 orders each and
R$260 lifetime GMV against R$138 for one-timers — **1.9× the value**. Their AOV
is actually *lower* (R$123 vs R$138); the extra value comes from frequency, not
from bigger baskets.

Before drawing any conclusion from 3.0%, confirm it is not an artefact of keying
on the wrong column:

In [3]:
q("""
SELECT 'customer_id (WRONG — order key)' AS keyed_on,
       ROUND(100.0*AVG(CASE WHEN n>1 THEN 1.0 ELSE 0 END), 2) AS repeat_rate_pct
  FROM (SELECT customer_id, COUNT(DISTINCT order_id) AS n
          FROM order_base WHERE is_eligible=1 GROUP BY customer_id)
UNION ALL
SELECT 'customer_unique_id (correct)',
       ROUND(100.0*AVG(CASE WHEN n>1 THEN 1.0 ELSE 0 END), 2)
  FROM (SELECT customer_unique_id, COUNT(DISTINCT order_id) AS n
          FROM order_base WHERE is_eligible=1 GROUP BY customer_unique_id)
""")

,keyed_on,repeat_rate_pct
0,customer_id (WRONG — order key),0.0
1,customer_unique_id (correct),3.0


Keying on `customer_id` returns exactly **0.00%** — every "customer"
looks brand new, forever. The query runs without error and the number is
plausible enough to publish. This is the single most common mistake made on the
Olist dataset.

**However — 3.0% is a floor, not the true rate.** The observation window is
~21 usable months and most customers appear near its end, so a buyer acquired in
2018-07 has weeks, not months, to return. The honest statement is *"at most 3%
of customers returned **within the observed window**"*, and the real long-run
rate is somewhat higher.

## 2. How long do repeat buyers take to come back?

In [4]:
pd.read_csv(ROOT / "outputs" / "tables" / "repeat_latency.csv")

,repeat_customers,avg_days_to_second_order,pct_within_30d,pct_within_90d,pct_within_180d
0,2801,81.2,50.4,68.1,82.7


Among customers who do return, **half come back within 30 days** and
83% within 180. The second purchase, when it happens at all, happens *soon*.

That sharpens the censoring argument above and also points at the intervention
window: a retention programme has roughly a one-month window after delivery, not
a one-year one.

In [5]:
q("""
WITH seq AS (
  SELECT customer_unique_id, gmv,
         ROW_NUMBER() OVER (PARTITION BY customer_unique_id ORDER BY order_purchase_timestamp) AS order_seq
    FROM order_base WHERE is_eligible=1)
SELECT order_seq, COUNT(*) AS orders, ROUND(AVG(gmv),2) AS avg_order_value
  FROM seq WHERE order_seq <= 4 GROUP BY order_seq ORDER BY order_seq
""")

,order_seq,orders,avg_order_value
0,1,93358,137.50
1,2,2801,123.54
2,3,228,121.42
3,4,47,163.38


The second order is *smaller* than the first (R$124 vs R$138), so
repeat value accrues through frequency. A retention offer sized against
first-order AOV would be over-generous.

## 3. Where the money actually sits

![customer value and segments](../outputs/figures/07_customer_value_and_segments.png)

In [6]:
pd.read_csv(ROOT / "outputs" / "tables" / "customer_value_deciles.csv")

,decile,customers,gmv,pct_of_gmv,cumulative_pct_of_gmv
0,1,9336,5434024.58,41.10,41.10
1,2,9336,2052212.48,15.52,56.62
2,3,9336,1462923.09,11.06,67.69
3,4,9336,1145909.82,8.67,76.35
4,5,9336,920254.29,6.96,83.31
5,6,9336,730177.31,5.52,88.84
6,7,9336,564365.95,4.27,93.10
7,8,9336,433902.45,3.28,96.39
8,9,9335,303650.24,2.30,98.68
9,10,9335,174077.90,1.32,100.00


**The top 10% of customers carry 41% of GMV; the top 30% carry 68%.**
Steep — but with a 3% repeat rate this concentration is driven mostly by
*order size*, not purchase frequency. These are largely one-time buyers of
expensive items, which is a different commercial problem from classic
high-frequency VIPs, and it calls for a different response.

## 4. RFM segmentation

Recency, Frequency and Monetary value, scored 1–5.

**One deliberate departure from textbook RFM.** Frequency is scored by a
business rule, not by `NTILE(5)`. With 97% of customers at exactly one order,
quintiles of frequency are degenerate: `NTILE` would split tied customers across
score boundaries arbitrarily, handing two identical one-time buyers different
scores purely by row order. The rule (`1 order → 1`, `2 → 3`, `3 → 4`, `4+ → 5`)
is reproducible and interpretable. Recency and Monetary keep their quintiles,
where the distributions are genuinely continuous.

In [7]:
pd.read_csv(ROOT / "outputs" / "tables" / "rfm_segments.csv")

,segment,customers,pct_of_customers,gmv,pct_of_gmv,gmv_per_customer,avg_recency_days,avg_orders
0,Promising high-value,14470,15.50,3865048.92,29.23,267.11,139.0,1.00
1,At risk — high value,13809,14.79,3820964.15,28.90,276.70,442.0,1.00
2,Occasional,18070,19.36,2346618.78,17.75,129.86,267.0,1.00
3,Dormant,22536,24.14,1257997.99,9.51,55.82,443.0,1.00
4,Recent one-time,21672,23.21,1202459.52,9.09,55.48,137.0,1.00
5,Champions,931,1.00,305122.29,2.31,327.74,136.0,2.19
6,Repeat — lapsing,997,1.07,248766.51,1.88,249.52,429.0,2.08
7,Loyal,873,0.94,174519.95,1.32,199.91,224.0,2.07


The result is an unusual shape, and the unusual part is the finding:

- **Champions / Loyal / Repeat-lapsing together are 3% of customers and 5.5% of GMV.**
  The segments a retention programme would normally target barely exist here.
- **"Promising high-value" (16% of customers) and "At risk — high value" (15%)
  hold 58% of GMV between them** — and both are one-time buyers. The difference
  between them is purely recency: one bought recently, one has not been seen in
  ~14 months.

So the actionable segment is not a loyal core to nurture. It is a large block of
**recent, high-spending, single-purchase customers** who have not yet been given
a reason to return — and an equally large block that has already gone quiet.
Winning back the second group is a far harder sell than converting the first,
which is why the recommendation prioritises the recent cohort.

In [8]:
q("""
SELECT segment, COUNT(*) AS customers, ROUND(AVG(recency_days),0) AS avg_recency_days,
       ROUND(AVG(gmv),2) AS avg_gmv, ROUND(AVG(orders),2) AS avg_orders
  FROM customer_rfm GROUP BY segment ORDER BY avg_gmv DESC
""")

,segment,customers,avg_recency_days,avg_gmv,avg_orders
0,Champions,931,136.0,327.74,2.19
1,At risk — high value,13809,442.0,276.70,1.00
2,Promising high-value,14470,139.0,267.11,1.00
3,Repeat — lapsing,997,429.0,249.52,2.08
4,Loyal,873,224.0,199.91,2.07
5,Occasional,18070,267.0,129.86,1.00
6,Dormant,22536,443.0,55.82,1.00
7,Recent one-time,21672,137.0,55.48,1.00


## 5. Growth is acquisition, not retention

![new vs returning](../outputs/figures/08_new_vs_returning.png)

In [9]:
nvr = pd.read_csv(ROOT / "outputs" / "tables" / "new_vs_returning_monthly.csv")
nvr.tail(8)

,order_month,new_customers,returning_customers,returning_gmv,returning_gmv_pct
12,2018-01,6842,132,17911.65,1.94
13,2018-02,6288,112,13383.95,1.62
14,2018-03,6774,140,16163.86,1.70
15,2018-04,6582,162,20868.61,2.14
16,2018-05,6506,187,24404.64,2.50
17,2018-06,5878,183,25378.11,2.96
18,2018-07,5949,151,20008.90,2.31
19,2018-08,6144,166,18517.56,2.21


Returning customers peak at **2.96% of monthly GMV** (2018-06) and sit
at 2.2% in the final full month. Growth through the whole period is new-customer
acquisition almost exclusively.

That is the strategic headline: GMV roughly plateaued across 2018 (see
`outputs/figures/01_monthly_gmv_orders.png`) *while* acquisition stayed strong —
which is what a business looks like when it has no retention engine.

## 6. Delivery reliability

![on-time trend](../outputs/figures/05_ontime_rate_trend.png)

In [10]:
pd.read_csv(ROOT / "outputs" / "tables" / "delivery_headline.csv")

,delivered_with_dates,on_time,late,on_time_pct,late_pct,avg_days_to_deliver,avg_days_early_vs_estimate
0,96470,89936,6534,93.23,6.77,12.56,11.88


**93.2% on time, and orders arrive on average 11.9 days *before* the
promised date.** The estimate is padded heavily — and that padding is itself a
finding: a 12.6-day average delivery against a ~24-day promise is a conservative
promise doing a lot of work to protect the on-time number.

The aggregate hides the real problem, which is variance:

In [11]:
ot = pd.read_csv(ROOT / "outputs" / "tables" / "ontime_monthly.csv")
ot.loc[ot.on_time_pct.nsmallest(4).index].sort_values("order_month")

,order_month,delivered_orders,on_time_pct,avg_days_to_deliver,avg_review
10,2017-11,7288,87.60,15.2,3.989
11,2017-12,5513,92.54,15.4,4.090
13,2018-02,6555,85.87,16.9,3.882
14,2018-03,7003,81.04,16.3,3.813


On-time rate swings from **98.8% (2018-06) to 81.0% (2018-03)** — an
18-point range. Three bad months (2017-11, 2018-02, 2018-03) account for a large
share of all late orders. 2017-11 is Black Friday; February–March 2018 has no
obvious seasonal cause in this data and is the more interesting failure.

An annual on-time KPI of 93% would conceal all of this. Reliability here is a
**variance** problem, not a level problem.

## 7. Delivery and satisfaction

![delivery vs satisfaction](../outputs/figures/04_delivery_vs_satisfaction.png)

In [12]:
pd.read_csv(ROOT / "outputs" / "tables" / "ontime_vs_late_reviews.csv")

,delivery_outcome,orders,avg_review,pct_1_or_2_star,pct_4_or_5_star
0,late,6381,2.272,62.36,26.74
1,on time,89443,4.291,9.23,82.64


In [13]:
pd.read_csv(ROOT / "outputs" / "tables" / "lateness_buckets.csv")

,bucket,bucket_min_days,orders,pct_of_orders,avg_review
0,15+ days early,-147.0,34939,36.22,4.323
1,8-14 days early,-14.0,36364,37.69,4.311
2,1-7 days early,-7.0,17341,17.98,4.201
3,on estimated day,0.0,1292,1.34,4.036
4,1-7 days late,1.0,3672,3.81,2.715
5,8-14 days late,8.0,1478,1.53,1.672
6,15-30 days late,15.0,1039,1.08,1.618
7,30+ days late,31.0,345,0.36,2.056


This is the strongest association in the dataset.

- On-time orders average **4.29**; late orders average **2.27** — a 2.0-point gap.
- **54% of late orders are rated 1 star**, against 7% of on-time orders.
- The drop is a **cliff, not a slope**: 4.04 on the estimated day → 2.71 at
  1–7 days late. Being a single day late costs more than a week of earliness buys.
- Earliness has little upside: 15+ days early scores 4.32 vs 4.20 for 1–7 days
  early. Padding estimates protects against the cliff; it does not itself
  generate satisfaction.

**This is an association, not a causal estimate.** Late deliveries are not
randomly assigned — they concentrate in distant states, heavier categories and
specific sellers, each of which could independently depress scores. The
consistent dose-response across buckets is suggestive, but nothing here
identifies a causal effect, and the 2.0-point gap should not be read as
"fixing lateness adds 2.0 stars".

In [14]:
pd.read_csv(ROOT / "outputs" / "tables" / "first_order_outcome_vs_repeat.csv")

,first_experience,customers,came_back,repeat_rate_pct
0,first order late,6353,158,2.49
1,first order on time,86997,2643,3.04


Customers whose **first** order arrived late return at 2.49% versus
3.04% for on-time — about **18% lower**. Directionally consistent with the
satisfaction gap, but both rates are small and the same confounders apply, so
treat it as a weak signal rather than a quantified retention cost.

## 8. Where delays concentrate

![state delay vs review](../outputs/figures/06_state_delay_vs_review.png)

In [15]:
ds = pd.read_csv(ROOT / "outputs" / "tables" / "delays_by_state.csv")
ds.head(10)

,customer_state,delivered_orders,late_orders,late_pct,avg_days_to_deliver,avg_freight,avg_review,gmv
0,SP,40494,1820,4.49,8.8,17.33,4.247,5066562.98
1,RJ,12350,1495,12.11,15.3,23.95,3.965,1759651.13
2,MG,11354,519,4.57,12.0,23.46,4.192,1552481.83
3,BA,3256,396,12.16,19.3,29.96,3.930,493584.14
4,RS,5344,325,6.08,15.3,24.81,4.186,728718.47
5,SC,3546,291,8.21,15.0,24.85,4.133,507012.13
6,ES,1995,214,10.73,15.8,24.57,4.080,268643.45
7,PR,4923,199,4.04,12.0,23.49,4.240,666063.51
8,CE,1279,176,13.76,21.3,36.50,3.942,219757.38
9,PE,1593,153,9.60,18.4,35.83,4.083,251889.49


Ranked by **absolute late orders**, not late rate — a 21% late rate on
397 orders (AL) matters less operationally than 12% on 12,350 (RJ).

**Rio de Janeiro is the clear priority.** It is the second-largest market by
volume yet has a 12.1% late rate — nearly triple São Paulo's 4.5% — and the
lowest average review of any high-volume state. RJ alone accounts for 1,495 late
orders, 23% of all late deliveries nationally, from 12.8% of orders.

The far-north and north-east states (MA, AL, PI, SE, CE) have worse *rates* and
much higher freight, which is consistent with distance. RJ is not a distance
story: its average delivery is 15.3 days against São Paulo's 8.8, over
comparable distances, which points at something structural — carrier coverage or
seller concentration — rather than geography.

In [16]:
pd.read_csv(ROOT / "outputs" / "tables" / "delays_by_category.csv").head(10)

,category,delivered_orders,late_pct,avg_days_to_deliver,avg_review,gmv
0,baby,2751,8.14,12.7,4.123,399850.15
1,office_furniture,1251,8.07,20.7,3.645,268979.97
2,electronics,2503,7.67,12.9,4.130,155368.72
3,musical_instruments,608,7.57,13.1,4.243,184469.81
4,health_beauty,8613,7.54,12.1,4.233,1233817.51
5,bed_bath_table,9175,7.47,13.0,4.008,1023899.79
6,construction_tools_construction,727,7.43,10.9,4.146,142377.81
7,watches_gifts,5475,7.42,12.8,4.123,1166619.18
8,auto,3789,7.34,12.3,4.153,578505.81
9,unknown,1361,7.27,12.8,4.043,169642.04


Category spread is narrow (4.2%–8.1%), so lateness is mostly a
*geography* and *carrier* problem rather than a product problem. The one
standout is **office furniture**: 20.7 days average delivery against a 12.6-day
overall mean, and the lowest review score in the table at 3.65 — a bulky-goods
fulfilment issue worth isolating.

## 9. What this implies

| Finding | Evidence | Implication |
| --- | --- | --- |
| Repeat rate ≤3% in-window | `customer_type_summary` | No retention engine exists; largest untapped lever |
| 58% of GMV from one-time high-value buyers | `rfm_segments` | Target the *recent* half first — recency is the only thing separating the two blocks |
| Half of returns happen within 30 days | `repeat_latency` | Post-delivery is the intervention window |
| On-time swings 81%–99% | `ontime_monthly` | Manage variance, not the annual average |
| Late → 2.27 stars vs 4.29 | `ontime_vs_late_reviews` | Lateness is the dominant satisfaction driver, directionally |
| RJ = 23% of late orders | `delays_by_state` | Highest-leverage single operational fix |
| 90% of orders are single-item | notebook 01 | Attach rate, not discounting, is the AOV lever |

Written up with recommendations in [`reports/executive_summary.md`](../reports/executive_summary.md).

In [17]:
conn.close()